In [1]:
# Import required libraries.
import pandas as pd
from IPython.display import display  # noqa: A004

# Dataset


In [2]:
hh_df = pd.read_csv("data/faps_household_puf.csv")

item_df = pd.read_csv("data/faps_fahitem_puf.csv")
nutrient_df = pd.read_csv("data/faps_fahnutrients.csv")

ID_VARIABLES = ["hhnum", "eventid", "itemnum"]
REQUIRED_VARIABLES = [*ID_VARIABLES, "totgramsedible", "totsug"]

food_df = item_df.merge(nutrient_df[REQUIRED_VARIABLES], how="left", on=ID_VARIABLES)

# Data Preprocessing


In [3]:
import enum
import warnings

hh_df = hh_df[(hh_df["numguests"] == 0) & (hh_df["hhsizechange"] == 0)]


class SNAPNowAdmin(enum.IntEnum):
    """Results from match of household members with SNAP administrative data."""

    NO_MATCH = 0
    """No match."""
    CONFIRMED_SNAP = 1
    """Match confirms SNAP participation."""
    CONFIRMED_NON_SNAP = 2
    """Match confirms SNAP nonparticipation."""
    VALID_SKIP = -996
    """Valid skip. (Consent for data matching not given.)"""


hh_df = hh_df[
    (hh_df["snapnowadmin"] == SNAPNowAdmin.CONFIRMED_SNAP) | (hh_df["snapnowadmin"] == SNAPNowAdmin.CONFIRMED_NON_SNAP)
]

with warnings.catch_warnings():
    # Ignore the `PerformanceWarning`
    warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

    hh_df["snap_status"] = (hh_df["snapnowadmin"] == SNAPNowAdmin.CONFIRMED_SNAP).astype("int8")

hh_df["region"] = hh_df["region"].map({1: "NORTHEAST", 2: "MIDWEST", 3: "SOUTH", 4: "WEST"})

food_df = food_df[(food_df["totgramsedible"] > 0) & (food_df["totsug"].notna())]

In [4]:
food_df["sugar_amount"] = (food_df["totsug"] / 100) * food_df["totgramsedible"]

HIGH_SUGAR_THRESHOLD = 22.5

food_df["high_sugar_status"] = (food_df["totsug"] > HIGH_SUGAR_THRESHOLD).astype("int8")

In [5]:
hh_sugar_df = (
    food_df.groupby("hhnum")
    .agg(total_sugar_amount=("sugar_amount", "sum"), total_amount=("totgramsedible", "sum"))
    .reset_index()
)
hh_sugar_df["sugar_share"] = hh_sugar_df["total_sugar_amount"] / hh_sugar_df["total_amount"]

# Add the household number and the SNAP status back for identification and categorization.
hh_sugar_df = hh_sugar_df.merge(hh_df[["hhnum", "snap_status", "region"]], how="left", on="hhnum")

# Data Mining


In [6]:
merged_df = food_df.merge(hh_df[["hhnum", "snap_status", "region"]], how="left", on="hhnum")

In [7]:
hh_transactions = (
    merged_df.groupby("hhnum")
    .agg({"high_sugar_status": lambda s: int((s == 1).any()), "snap_status": "first", "region": "first"})
    .reset_index()
)

In [8]:
basket = hh_transactions.copy()
basket["SNAP"] = (basket["snap_status"] == 1).astype("int8")
basket["HIGH_SUGAR"] = basket["high_sugar_status"]

region_dummies = pd.get_dummies(basket["region"], prefix="REGION", dtype="int8")

basket = pd.concat([basket, region_dummies], axis=1)
basket = basket.drop(columns=["hhnum", "snap_status", "region", "high_sugar_status"])

basket

,SNAP,HIGH_SUGAR,REGION_MIDWEST,REGION_NORTHEAST,REGION_SOUTH,REGION_WEST
0,1,1,0,0,1,0
1,0,1,0,0,0,0
2,0,1,0,0,0,0
3,0,0,0,0,0,0
4,0,1,0,0,0,0
...,...,...,...,...,...,...
4230,0,1,0,0,0,0
4231,0,1,0,0,0,0
4232,1,1,0,0,1,0
4233,1,0,0,0,1,0


In [9]:
from mlxtend.frequent_patterns import apriori

with warnings.catch_warnings():
    # Ignore the `DeprecationWarning`
    warnings.filterwarnings("ignore", category=DeprecationWarning)

    frequent_itemsets = apriori(basket, min_support=0.05, use_colnames=True)

In [10]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
rules[(rules["antecedents"].apply(lambda x: "SNAP" in x)) & (rules["consequents"].apply(lambda x: "HIGH_SUGAR" in x))]

rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({SNAP}),frozenset({REGION_SOUTH}),0.238489,0.117591,0.105785,0.443564,3.772078,1.0,0.077741,1.585823,0.965047,0.422642,0.369413,0.671581
1,frozenset({REGION_SOUTH}),frozenset({SNAP}),0.117591,0.238489,0.105785,0.899598,3.772078,1.0,0.077741,7.584652,0.832828,0.422642,0.868155,0.671581
2,frozenset({SNAP}),frozenset({REGION_WEST}),0.238489,0.063754,0.058087,0.243564,3.820352,1.0,0.042883,1.237707,0.969446,0.237911,0.192054,0.577338
3,frozenset({REGION_WEST}),frozenset({SNAP}),0.063754,0.238489,0.058087,0.911111,3.820352,1.0,0.042883,8.567001,0.788515,0.237911,0.883273,0.577338
4,frozenset({HIGH_SUGAR}),frozenset({REGION_SOUTH}),0.701771,0.117591,0.083589,0.119112,1.012928,1.0,0.001067,1.001726,0.042796,0.113607,0.001723,0.414978
5,frozenset({REGION_SOUTH}),frozenset({HIGH_SUGAR}),0.117591,0.701771,0.083589,0.710843,1.012928,1.0,0.001067,1.031375,0.014464,0.113607,0.030421,0.414978
6,"frozenset({HIGH_SUGAR, SNAP})",frozenset({REGION_SOUTH}),0.166706,0.117591,0.075325,0.451841,3.842466,1.0,0.055721,1.609768,0.887742,0.360452,0.378793,0.546202
7,"frozenset({HIGH_SUGAR, REGION_SOUTH})",frozenset({SNAP}),0.083589,0.238489,0.075325,0.901130,3.778500,1.0,0.055390,7.702142,0.802418,0.305263,0.870166,0.608486
8,"frozenset({SNAP, REGION_SOUTH})",frozenset({HIGH_SUGAR}),0.105785,0.701771,0.075325,0.712054,1.014652,1.0,0.001088,1.035710,0.016149,0.102870,0.034479,0.409694
9,frozenset({HIGH_SUGAR}),"frozenset({SNAP, REGION_SOUTH})",0.701771,0.105785,0.075325,0.107335,1.014652,1.0,0.001088,1.001736,0.048422,0.102870,0.001733,0.409694


# Statistical Inference


## Independent t-Test


In [11]:
from scipy.stats import ttest_ind

snap = hh_sugar_df[hh_sugar_df["snap_status"] == 1]["sugar_share"]
non_snap = hh_sugar_df[hh_sugar_df["snap_status"] == 0]["sugar_share"]

t_stat, p_val = ttest_ind(snap, non_snap, equal_var=False)

ALPHA = 0.05

print(f"t-statistic = {t_stat:4f}")
print(f"p-value     = {p_val:4f}")
print()
print(f"Result: {'Reject H0' if p_val < ALPHA else 'Fail to reject H0'}")

t-statistic = -0.300006
p-value     = 0.764680

Result: Fail to reject H0


## Two-Way ANOVA


In [12]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

model = smf.ols("sugar_share ~ C(snap_status) + C(region) + C(snap_status):C(region)", data=hh_sugar_df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

display(anova_table)

for factor in anova_table.index:
    p_val = anova_table.loc[factor, "PR(>F)"]

    print(f"{factor:<{24}}: {'Significant effect' if p_val < ALPHA else 'No significant effect'}")

,sum_sq,df,F,PR(>F)
C(snap_status),0.000145,1.0,0.035948,0.849658
C(region),0.045037,3.0,3.726256,0.011055
C(snap_status):C(region),0.025168,3.0,2.082288,0.100847
Residual,4.463952,1108.0,NaN,NaN


C(snap_status)          : No significant effect
C(region)               : Significant effect
C(snap_status):C(region): No significant effect
Residual                : No significant effect


## Chi-Square Test of Independence


In [13]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(hh_transactions["snap_status"], hh_transactions["high_sugar_status"])
chi2, p_val, dof, expected = chi2_contingency(contingency)

display(contingency)

expected_df = pd.DataFrame(expected, index=contingency.index, columns=contingency.columns)

display(expected_df)

print(f"x^2-stat = {chi2:.4f}")
print(f"p-value  = {p_val:.4f}")
print(f"d f      = {dof}")
print()
print(f"Result: {'Reject H0' if p_val < ALPHA else 'Fail to reject H0'}")

high_sugar_status,0,1
snap_status,,
0.0,37,69
1.0,304,706


high_sugar_status,0,1
snap_status,,
0.0,32.388889,73.611111
1.0,308.611111,701.388889


x^2-stat = 0.8303
p-value  = 0.3622
d f      = 1

Result: Fail to reject H0
